# CodeBERT + Deep Classification Head (Task A)

**Architecture:** CodeBERT [CLS] embedding (768-d) concatenated with 8 handcrafted
stylometric features → deep MLP head `776 → 256 → 128 → 64 → 2` with GELU activations
and moderate dropout.

**Training recipe:** 40K stratified samples, 80/10/10 split, Adam + cosine LR scheduler
with warmup, gradient clipping, mixed-precision (fp16), early stopping, model checkpointing,
weight decay, optional layer-wise LR decay.

**Evaluation:** FULL held-out test set only.

In [ ]:
!pip install --upgrade transformers==4.45.0 huggingface_hub

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

import re
import math
import hashlib
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from collections import Counter
from tqdm import tqdm
from datasets import Dataset
from transformers import (
    RobertaTokenizer,
    RobertaModel,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
    get_cosine_schedule_with_warmup,
)
from transformers.modeling_outputs import SequenceClassifierOutput
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
)
import warnings
warnings.filterwarnings("ignore")

print("All imports ready.")

## 1 · Data Loading & Inline Preprocessing

Applies the same cleaning pipeline as `clean_and_flag_task_a.py`:
- Drop missing / placeholder rows
- Flag & remove encoding-suspect rows
- Exact deduplication via SHA-1 of whitespace-normalised text
- Remove contradictory-label duplicates

In [ ]:
# ============================================================================
#  Inline preprocessing  (mirrors clean_and_flag_task_a.py)
# ============================================================================

PLACEHOLDERS = {"unk", "na", "n/a", "-", "?", "none", "null", "nan"}
ENCODING_RE  = re.compile(r"Ã.|â\u20ac™|â\u20acœ|â\u20ac|\ufffd|\?\?\?")


def normalize_ws(s: pd.Series) -> pd.Series:
    return s.fillna("").astype(str).str.replace(r"\s+", " ", regex=True).str.strip()


def sha1_series(s: pd.Series) -> pd.Series:
    return s.map(lambda x: hashlib.sha1(x.encode("utf-8", "ignore")).hexdigest())


def clean_task_a(df: pd.DataFrame, text_col: str = "code",
                 label_col: str = "label") -> pd.DataFrame:
    """
    Apply the same cleaning steps as clean_and_flag_task_a.py:
      1. Drop rows with missing / placeholder code
      2. Drop encoding-suspect rows
      3. Exact deduplication (SHA-1 of whitespace-normalised text)
      4. Drop contradictory-label groups (same code → different labels)
    """
    n0 = len(df)

    # --- Drop missing-like rows ---
    raw = df[text_col]
    is_null     = raw.isna()
    is_empty    = raw.fillna("").astype(str).eq("")
    is_ws       = raw.fillna("").astype(str).str.fullmatch(r"\s*")
    is_placeholder = raw.fillna("").astype(str).str.strip().str.lower().isin(PLACEHOLDERS)
    missing_mask = is_null | is_empty | is_ws | is_placeholder
    df = df[~missing_mask].reset_index(drop=True)
    print(f"  Dropped {missing_mask.sum()} missing/placeholder rows")

    # --- Drop encoding-suspect rows ---
    text_norm = normalize_ws(df[text_col])
    enc_mask = text_norm.str.contains(ENCODING_RE)
    n_enc = enc_mask.sum()
    df = df[~enc_mask].reset_index(drop=True)
    print(f"  Dropped {n_enc} encoding-suspect rows")

    # --- Exact deduplication ---
    text_norm = normalize_ws(df[text_col])
    exact_hash = sha1_series(text_norm)
    dup_mask = exact_hash.duplicated(keep="first")
    n_dup = dup_mask.sum()
    df = df[~dup_mask].reset_index(drop=True)
    print(f"  Dropped {n_dup} exact duplicates")

    # --- Drop contradictory-label groups ---
    text_norm = normalize_ws(df[text_col])
    exact_hash = sha1_series(text_norm)
    grp = pd.DataFrame({"h": exact_hash, "y": df[label_col].astype(str)})
    bad_hashes = set(grp.groupby("h")["y"].nunique().pipe(lambda s: s[s > 1]).index)
    contra_mask = exact_hash.isin(bad_hashes)
    n_contra = contra_mask.sum()
    df = df[~contra_mask].reset_index(drop=True)
    print(f"  Dropped {n_contra} contradictory-label rows")

    print(f"  Cleaning: {n0} → {len(df)} rows")
    return df


print("Preprocessing functions defined.")

In [ ]:
# ============================================================================
#  Load training parquet, clean, stratified-sample 40K, 80/10/10 split
# ============================================================================

SAMPLE_SIZE = 40_000
RANDOM_SEED = 42

raw_df = pd.read_parquet(
    "/kaggle/input/datasets/daniilor/semeval-2026-task13/"
    "SemEval-2026-Task13/task_a/task_a_training_set_1.parquet"
)
print(f"Raw dataset: {len(raw_df)} rows, columns: {raw_df.columns.tolist()}")
raw_df["label"] = raw_df["label"].astype(int)

# --- Clean ---
df_clean = clean_task_a(raw_df, text_col="code", label_col="label")

# --- Stratified sample of 40K ---
if SAMPLE_SIZE < len(df_clean):
    df_sampled = df_clean.groupby("label", group_keys=False).apply(
        lambda x: x.sample(
            n=max(1, int(SAMPLE_SIZE * len(x) / len(df_clean))),
            random_state=RANDOM_SEED,
        )
    ).reset_index(drop=True)
else:
    df_sampled = df_clean.copy()
print(f"Sampled: {len(df_sampled)} rows")
print(df_sampled["label"].value_counts().sort_index())

# --- 80 / 10 / 10 split (stratified) ---
train_df, temp_df = train_test_split(
    df_sampled, test_size=0.20, stratify=df_sampled["label"],
    random_state=RANDOM_SEED,
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label"],
    random_state=RANDOM_SEED,
)

print(f"\nSplits — Train: {len(train_df)},  Val: {len(val_df)},  Test: {len(test_df)}")
print(f"Train labels:\n{train_df['label'].value_counts().sort_index()}")
print(f"Val labels:\n{val_df['label'].value_counts().sort_index()}")
print(f"Test labels:\n{test_df['label'].value_counts().sort_index()}")

## 2 · Handcrafted Code Features (8 stylometric signals)

In [ ]:
# ============================================================================
#  8 language-agnostic stylometric features  (identical to final_model.ipynb)
# ============================================================================

NUM_CODE_FEATURES = 8


def extract_code_features(code: str) -> list:
    """
    Returns 8 float values in roughly [0, 1]:
      1. comment_ratio
      2. avg_line_length  (normalised)
      3. line_length_cv
      4. whitespace_consistency  (normalised)
      5. max_nesting_depth  (normalised)
      6. blank_line_ratio
      7. char_entropy  (Shannon, normalised)
      8. bracket_density  (normalised)
    """
    lines = code.split("\n")
    non_empty = [l for l in lines if l.strip()]
    total_lines = max(len(lines), 1)
    ne_count = max(len(non_empty), 1)

    # 1
    comment_lines = sum(
        1 for l in lines
        if l.strip().startswith("#") or l.strip().startswith("//")
    )
    comment_ratio = comment_lines / total_lines

    # 2
    lengths = [len(l) for l in non_empty]
    mean_len = sum(lengths) / ne_count if lengths else 0.0
    avg_line_len = min(mean_len / 200.0, 1.0)

    # 3
    if lengths and mean_len > 0:
        var = sum((x - mean_len) ** 2 for x in lengths) / ne_count
        cv = math.sqrt(var) / mean_len
    else:
        cv = 0.0
    line_len_cv = min(cv / 2.0, 1.0)

    # 4
    leading = [len(l) - len(l.lstrip()) for l in non_empty]
    if leading:
        m = sum(leading) / len(leading)
        ws_var = sum((x - m) ** 2 for x in leading) / len(leading)
        ws_std = math.sqrt(ws_var)
    else:
        ws_std = 0.0
    ws_consistency = min(ws_std / 20.0, 1.0)

    # 5
    max_indent = max(leading) if leading else 0
    max_depth = min(max_indent / 32.0, 1.0)

    # 6
    blank_lines = sum(1 for l in lines if not l.strip())
    blank_line_ratio = blank_lines / total_lines

    # 7
    if len(code) > 0:
        char_counts = Counter(code)
        total_chars = len(code)
        entropy = -sum(
            (c / total_chars) * math.log2(c / total_chars)
            for c in char_counts.values()
        )
        char_entropy = min(entropy / 7.0, 1.0)
    else:
        char_entropy = 0.0

    # 8
    brackets = sum(1 for c in code if c in "()[]{}")
    bracket_density = min(brackets / max(len(code), 1) * 20.0, 1.0)

    return [
        comment_ratio, avg_line_len, line_len_cv,
        ws_consistency, max_depth,
        blank_line_ratio, char_entropy, bracket_density,
    ]


print("extract_code_features() defined.")

## 3 · Model Architecture

Deep classification head: `[CLS](768) ⊕ features(8) = 776 → 256 → 128 → 64 → 2`
with GELU activations, LayerNorm, and dropout (~0.1) between layers.

In [ ]:
# ============================================================================
#  CodeBERT + Deep Classification Head  (776 → 256 → 128 → 64 → 2)
# ============================================================================

class DeepHeadCodeBERT(nn.Module):
    """
    CodeBERT [CLS] (768-d) concatenated with 8 handcrafted features
    → deep MLP: 776 → 256 → 128 → 64 → 2
    Uses GELU activations, LayerNorm, and moderate dropout.
    """

    def __init__(
        self,
        model_name: str = "microsoft/codebert-base",
        num_labels: int = 2,
        num_features: int = NUM_CODE_FEATURES,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.num_labels = num_labels

        # --- Transformer backbone ---
        self.transformer = RobertaModel.from_pretrained(model_name)
        self.config = self.transformer.config
        hidden_size = self.config.hidden_size  # 768

        # --- Feature normalisation ---
        self.feat_norm = nn.LayerNorm(num_features)

        # --- Deep classification head ---
        input_dim = hidden_size + num_features  # 776
        self.head = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(128, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(64, num_labels),
        )

        # Initialise head weights (Xavier)
        for m in self.head:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        labels=None,
        code_features=None,
        **kwargs,
    ):
        out = self.transformer(
            input_ids=input_ids, attention_mask=attention_mask
        )
        cls_vec = out.last_hidden_state[:, 0, :]  # (B, 768)

        if code_features is not None:
            feat = self.feat_norm(code_features.float())
            combined = torch.cat([cls_vec, feat], dim=-1)  # (B, 776)
        else:
            combined = torch.cat(
                [cls_vec,
                 cls_vec.new_zeros(cls_vec.size(0), self.feat_norm.normalized_shape[0])],
                dim=-1,
            )

        logits = self.head(combined)

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)

        return SequenceClassifierOutput(loss=loss, logits=logits)


print("DeepHeadCodeBERT defined.")
print("  Architecture: [CLS](768) ⊕ features(8) → 256 → 128 → 64 → 2")

## 4 · Data Collator & Dataset Preparation

In [ ]:
# ============================================================================
#  Data collator that stacks code_features alongside the tokenised batch
# ============================================================================

class FeaturesDataCollator:
    """Wraps DataCollatorWithPadding; additionally stacks code_features."""

    def __init__(self, tokenizer):
        self.base = DataCollatorWithPadding(tokenizer=tokenizer)

    def __call__(self, features):
        code_feats = None
        if "code_features" in features[0]:
            code_feats = [f.pop("code_features") for f in features]
        batch = self.base(features)
        if code_feats is not None:
            batch["code_features"] = torch.tensor(
                code_feats, dtype=torch.float32
            )
        return batch


print("FeaturesDataCollator defined.")

In [ ]:
# ============================================================================
#  Tokeniser + HuggingFace Datasets
# ============================================================================

MODEL_NAME = "microsoft/codebert-base"
MAX_LENGTH = 512

tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)


def tokenize_fn(examples):
    return tokenizer(
        examples["code"],
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )


def add_features(examples):
    return {"code_features": [extract_code_features(c) for c in examples["code"]]}


def make_hf_dataset(df: pd.DataFrame) -> Dataset:
    ds = Dataset.from_pandas(df[["code", "label"]].reset_index(drop=True))
    ds = ds.map(add_features, batched=True, desc="Extracting features")
    ds = ds.map(tokenize_fn, batched=True, remove_columns=["code"],
                desc="Tokenising")
    ds = ds.rename_column("label", "labels")
    return ds


train_dataset = make_hf_dataset(train_df)
val_dataset   = make_hf_dataset(val_df)
test_dataset  = make_hf_dataset(test_df)

print(f"HF datasets ready — Train: {len(train_dataset)}, "
      f"Val: {len(val_dataset)}, Test: {len(test_dataset)}")

## 5 · Training

Adam optimiser, cosine LR schedule with warmup, gradient clipping, fp16,
early stopping (patience 3), model checkpointing (best F1), weight decay.

In [ ]:
# ============================================================================
#  Layer-wise learning-rate decay  (LLRD)
# ============================================================================

def get_layer_wise_lr_groups(
    model: DeepHeadCodeBERT,
    base_lr: float = 2e-5,
    head_lr: float = 1e-3,
    weight_decay: float = 0.01,
    llrd_factor: float = 0.95,
):
    """
    Assign per-layer LRs:
      - Embedding layer gets base_lr * llrd_factor^N  (lowest LR)
      - Each encoder layer i gets base_lr * llrd_factor^(N-i)
      - Classification head gets head_lr  (highest LR)
    """
    opt_params = []
    no_decay = {"bias", "LayerNorm.weight", "LayerNorm.bias"}

    # --- Transformer encoder layers ---
    num_layers = model.config.num_hidden_layers  # 12 for codebert-base

    # Embeddings
    emb_params_wd = []
    emb_params_nowd = []
    for n, p in model.transformer.embeddings.named_parameters():
        if any(nd in n for nd in no_decay):
            emb_params_nowd.append(p)
        else:
            emb_params_wd.append(p)
    emb_lr = base_lr * (llrd_factor ** num_layers)
    if emb_params_wd:
        opt_params.append({"params": emb_params_wd, "lr": emb_lr,
                           "weight_decay": weight_decay})
    if emb_params_nowd:
        opt_params.append({"params": emb_params_nowd, "lr": emb_lr,
                           "weight_decay": 0.0})

    # Encoder layers
    for i in range(num_layers):
        layer = model.transformer.encoder.layer[i]
        layer_lr = base_lr * (llrd_factor ** (num_layers - i))
        wd_p, nowd_p = [], []
        for n, p in layer.named_parameters():
            if any(nd in n for nd in no_decay):
                nowd_p.append(p)
            else:
                wd_p.append(p)
        if wd_p:
            opt_params.append({"params": wd_p, "lr": layer_lr,
                               "weight_decay": weight_decay})
        if nowd_p:
            opt_params.append({"params": nowd_p, "lr": layer_lr,
                               "weight_decay": 0.0})

    # --- Classification head + feature norm ---
    head_wd, head_nowd = [], []
    for module in [model.head, model.feat_norm]:
        for n, p in module.named_parameters():
            if any(nd in n for nd in no_decay):
                head_nowd.append(p)
            else:
                head_wd.append(p)
    if head_wd:
        opt_params.append({"params": head_wd, "lr": head_lr,
                           "weight_decay": weight_decay})
    if head_nowd:
        opt_params.append({"params": head_nowd, "lr": head_lr,
                           "weight_decay": 0.0})

    return opt_params


print("Layer-wise LR decay helper defined.")

In [ ]:
# ============================================================================
#  Custom Trainer that injects LLRD optimiser + cosine scheduler
# ============================================================================

class DeepHeadTrainer(Trainer):
    """Trainer subclass that uses layer-wise LR decay + cosine schedule."""

    def __init__(self, *args, llrd_factor=0.95, head_lr=1e-3, **kwargs):
        self.llrd_factor = llrd_factor
        self.head_lr = head_lr
        super().__init__(*args, **kwargs)

    def create_optimizer_and_scheduler(self, num_training_steps):
        param_groups = get_layer_wise_lr_groups(
            self.model,
            base_lr=self.args.learning_rate,
            head_lr=self.head_lr,
            weight_decay=self.args.weight_decay,
            llrd_factor=self.llrd_factor,
        )
        self.optimizer = torch.optim.AdamW(param_groups)

        warmup_steps = int(num_training_steps * self.args.warmup_ratio)
        self.lr_scheduler = get_cosine_schedule_with_warmup(
            self.optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=num_training_steps,
        )


print("DeepHeadTrainer defined.")

In [ ]:
# ============================================================================
#  Initialise model, run training
# ============================================================================

DROPOUT     = 0.1
NUM_EPOCHS  = 10
BATCH_SIZE  = 16
BASE_LR     = 2e-5
HEAD_LR     = 1e-3
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
GRAD_CLIP    = 1.0
LLRD_FACTOR  = 0.95
OUTPUT_DIR   = "/kaggle/working/codebert_deep_head"

# --- Model ---
model = DeepHeadCodeBERT(
    model_name=MODEL_NAME,
    num_labels=2,
    num_features=NUM_CODE_FEATURES,
    dropout=DROPOUT,
).to("cuda")

total_p = sum(p.numel() for p in model.parameters())
train_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model params: {total_p:,} total, {train_p:,} trainable")

# --- Metrics ---
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.argmax(preds, axis=1)
    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted"
    )
    return {"accuracy": acc, "f1": f1, "precision": prec, "recall": rec}

# --- Training args ---
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    weight_decay=WEIGHT_DECAY,
    logging_dir=f"{OUTPUT_DIR}/logs",
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    remove_unused_columns=False,
    learning_rate=BASE_LR,
    lr_scheduler_type="cosine",       # overridden by custom scheduler
    warmup_ratio=WARMUP_RATIO,
    max_grad_norm=GRAD_CLIP,
    save_total_limit=2,
    report_to=[],
    fp16=True,
)

data_collator = FeaturesDataCollator(tokenizer=tokenizer)

trainer = DeepHeadTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    llrd_factor=LLRD_FACTOR,
    head_lr=HEAD_LR,
)

print("Starting training ...")
trainer.train()

trainer.save_model()
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Training complete. Best model saved to {OUTPUT_DIR}")

## 6 · Evaluation (FULL held-out test set only)

In [ ]:
# ============================================================================
#  Test-set evaluation utilities  (4 category breakdown)
# ============================================================================

SEEN_LANGS     = {"c++", "cpp", "python", "java"}
UNSEEN_LANGS   = {"go", "php", "c#", "csharp", "c", "javascript", "js"}
SEEN_DOMAINS   = {"algorithmic"}
UNSEEN_DOMAINS = {"research", "production"}


def _norm(v):
    return str(v).strip().lower()


def evaluate_by_category(df, tag="Model"):
    """Print classification metrics for 4 evaluation settings + overall."""
    lang_col = next(
        (c for c in df.columns
         if c.lower() in ("language", "lang", "programming_language")),
        None,
    )
    domain_col = next(
        (c for c in df.columns
         if c.lower() in ("domain", "task_type", "category")),
        None,
    )

    if "label" not in df.columns:
        print(f"[{tag}] No 'label' column — cannot evaluate.")
        return

    if lang_col is None or domain_col is None:
        print(f"[{tag}] Missing language/domain columns. Overall only:")
        print(classification_report(df["label"], df["prediction"]))
        return

    df["_l"] = df[lang_col].apply(_norm)
    df["_d"] = df[domain_col].apply(_norm)

    settings = [
        ("(i)   Seen Lang & Seen Domain",    SEEN_LANGS,   SEEN_DOMAINS),
        ("(ii)  Unseen Lang & Seen Domain",   UNSEEN_LANGS, SEEN_DOMAINS),
        ("(iii) Seen Lang & Unseen Domain",   SEEN_LANGS,   UNSEEN_DOMAINS),
        ("(iv)  Unseen Lang & Unseen Domain", UNSEEN_LANGS, UNSEEN_DOMAINS),
    ]

    print(f"\n{'=' * 70}")
    print(f"  TEST RESULTS — {tag}")
    print(f"{'=' * 70}")

    for name, langs, domains in settings:
        mask = df["_l"].isin(langs) & df["_d"].isin(domains)
        sub = df[mask]
        n = len(sub)
        if n == 0:
            print(f"\n  {name}:  ** no samples **")
            continue
        y_t, y_p = sub["label"].values, sub["prediction"].values
        acc = accuracy_score(y_t, y_p)
        p, r, f1, _ = precision_recall_fscore_support(
            y_t, y_p, average="weighted", zero_division=0
        )
        print(f"\n  {name}  (n={n})")
        print(f"    Accuracy={acc:.4f}  Prec={p:.4f}  Recall={r:.4f}  F1={f1:.4f}")
        print(classification_report(y_t, y_p, zero_division=0))

    # Overall
    acc = accuracy_score(df["label"], df["prediction"])
    _, _, f1, _ = precision_recall_fscore_support(
        df["label"], df["prediction"], average="weighted", zero_division=0
    )
    print(f"\n  OVERALL  (n={len(df)})  Accuracy={acc:.4f}  F1={f1:.4f}")
    print("=" * 70)

    df.drop(columns=["_l", "_d"], inplace=True, errors="ignore")


print("Evaluation utilities defined.")

In [ ]:
# ============================================================================
#  Run inference on the FULL held-out test split
# ============================================================================

@torch.no_grad()
def predict_on_dataset(model, tokenizer, df, max_length=512, batch_size=32,
                       device="cuda"):
    model.to(device)
    model.eval()
    codes = df["code"].tolist()
    preds = []

    for i in tqdm(range(0, len(codes), batch_size), desc="Predicting"):
        batch = codes[i : i + batch_size]
        enc = tokenizer(
            batch, truncation=True, padding=True,
            max_length=max_length, return_tensors="pt",
        )
        fwd_kwargs = {
            "input_ids": enc["input_ids"].to(device),
            "attention_mask": enc["attention_mask"].to(device),
            "code_features": torch.tensor(
                [extract_code_features(c) for c in batch],
                dtype=torch.float32,
            ).to(device),
        }
        logits = model(**fwd_kwargs).logits
        preds.extend(logits.argmax(dim=-1).cpu().tolist())

    result = df.copy()
    result["prediction"] = preds
    return result


# --- Evaluate on the held-out test split ---
print(f"Evaluating on held-out test set ({len(test_df)} samples) ...")
test_results = predict_on_dataset(
    model, tokenizer, test_df,
    max_length=MAX_LENGTH, batch_size=32, device="cuda",
)

evaluate_by_category(test_results, tag="CodeBERT-DeepHead")

In [ ]:
# ============================================================================
#  Also evaluate on the FULL official test parquet (if available)
# ============================================================================

TEST_PARQUET = (
    "/kaggle/input/datasets/daniilor/semeval-2026-task13/"
    "SemEval-2026-Task13/task_a/task_a_test_set_sample.parquet"
)

if os.path.exists(TEST_PARQUET):
    print(f"\nEvaluating on official test set: {TEST_PARQUET}")
    official_test_df = pd.read_parquet(TEST_PARQUET)
    official_test_df = official_test_df.dropna(subset=["code"]).reset_index(drop=True)
    if "label" in official_test_df.columns:
        official_test_df["label"] = official_test_df["label"].astype(int)
    print(f"Official test set size: {len(official_test_df)}")

    official_results = predict_on_dataset(
        model, tokenizer, official_test_df,
        max_length=MAX_LENGTH, batch_size=32, device="cuda",
    )
    evaluate_by_category(official_results, tag="CodeBERT-DeepHead (Official Test)")
else:
    print(f"Official test parquet not found at {TEST_PARQUET}, skipping.")

In [ ]:
# ============================================================================
#  Summary
# ============================================================================

print("\n" + "=" * 70)
print("  CONFIGURATION SUMMARY")
print("=" * 70)
print(f"  Model:             {MODEL_NAME}")
print(f"  Head:              776 → 256 → 128 → 64 → 2  (GELU + LayerNorm)")
print(f"  Features:          {NUM_CODE_FEATURES} handcrafted stylometric")
print(f"  Sample size:       {SAMPLE_SIZE:,} (stratified from ~100K)")
print(f"  Split:             80/10/10 (train/val/test)")
print(f"  Epochs:            {NUM_EPOCHS}")
print(f"  Batch size:        {BATCH_SIZE}")
print(f"  Base LR:           {BASE_LR}")
print(f"  Head LR:           {HEAD_LR}")
print(f"  LLRD factor:       {LLRD_FACTOR}")
print(f"  Weight decay:      {WEIGHT_DECAY}")
print(f"  Warmup ratio:      {WARMUP_RATIO}")
print(f"  Grad clip:         {GRAD_CLIP}")
print(f"  Dropout:           {DROPOUT}")
print(f"  Precision:         fp16 (mixed)")
print(f"  Early stopping:    patience=3, metric=f1")
print(f"  Scheduler:         cosine with warmup")
print("=" * 70)